# Maturity Report

End-to-end runner for the maturity suite. Three sections:

1. **Regenerate fixtures** — rebuild the small hand-authored models and
   the scalability fixtures under `examples/maturity/`. Reproducible;
   safe to re-run.
2. **Scalability benchmark** — time the full pipeline (load → trace
   extraction → structural self-similarity) on the linear and AND
   shapes. Produces the numbers that go into the paper text.
3. **Edge-case scores** — run every pair the maturity tests exercise
   and print the actual sub-scores so thresholds can be calibrated and
   prose can quote the right numbers.

Run all cells top-to-bottom. The whole notebook completes in ~30 s on
a modern laptop.

## Setup

In [1]:
# Standard imports + repo path wiring. We add both the repo root and the
# `model_evaluation/` flat-layout module dir so the bpmn_* modules import
# cleanly when the notebook is opened from anywhere under the repo.

from __future__ import annotations

import sys
import time
import json
from pathlib import Path
from typing import Optional

import pandas as pd

# Resolve repo root from this notebook's location. The notebook lives in
# `notebooks/` — go up one level.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pyproject.toml").exists(), f"could not find repo root from {Path.cwd()}"

MODEL_EVAL = REPO_ROOT / "model_evaluation"
SCRIPTS_DIR = REPO_ROOT / "scripts"
for p in (str(REPO_ROOT), str(MODEL_EVAL), str(SCRIPTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

EXAMPLES = REPO_ROOT / "examples"
MATURITY_DIR = EXAMPLES / "maturity"
SCALABILITY_DIR = MATURITY_DIR / "scalability"

print(f"Repo root: {REPO_ROOT}")
print(f"Maturity fixtures: {MATURITY_DIR}")

Repo root: /Users/I762870/dev/process-evaluation-framework
Maturity fixtures: /Users/I762870/dev/process-evaluation-framework/examples/maturity


In [2]:
# Pipeline imports. These are the same modules the dashboard and the
# pytest suite use; the notebook is just a different orchestrator on top.

from BPMN_conversion import BPMNConverter, XMLBPMNConverter
from bpmn_similarity import (
    calculate_bpmn_similarity,
    calculate_trace_similarity,
)
from trace_extraction import extract_traces


def load_model(path: Path):
    # Mirror comparison_widget._load_model: XML/BPMN through the XML
    # converter, anything else as Signavio JSON.
    if path.suffix in (".xml", ".bpmn"):
        return XMLBPMNConverter.convert_file(str(path)).to_dict()
    with path.open("r", encoding="utf-8") as f:
        return BPMNConverter.convert(json.load(f)).to_dict()

## 1. Regenerate fixtures

Rebuild every generated BPMN file under `examples/maturity/`. The
generators are deterministic — running this cell never changes anything
that the suite hasn't already seen. Skip this section if you only want
to rerun the report against the current fixtures on disk.

In [3]:
# Regenerate the small hand-authored models (sanity rename pair,
# semantic-naming pairs, degenerate fixtures, subprocess pair).
from maturity.generate_small_models import emit_all as emit_small_models

small_written = emit_small_models(MATURITY_DIR)
print(f"Small models: wrote {len(small_written)} files")
for path in small_written:
    print(f"  {path.relative_to(REPO_ROOT)}")

Small models: wrote 10 files
  examples/maturity/sanity/renamed_only_a.bpmn
  examples/maturity/sanity/renamed_only_b.bpmn
  examples/maturity/semantic_naming/paraphrase_a.bpmn
  examples/maturity/semantic_naming/paraphrase_b.bpmn
  examples/maturity/semantic_naming/synonym_a.bpmn
  examples/maturity/semantic_naming/synonym_b.bpmn
  examples/maturity/degenerate/empty.bpmn
  examples/maturity/degenerate/single_task.bpmn
  examples/maturity/subprocess_folding/flat.bpmn
  examples/maturity/subprocess_folding/with_subprocess.bpmn


In [4]:
# Regenerate the scalability fixtures (linear N∈{5..200}, AND N∈{2..8}).
from maturity.generate_scalability_models import emit_all as emit_scalability_models

scalability_written = emit_scalability_models(SCALABILITY_DIR)
print(f"Scalability models: wrote {len(scalability_written)} files")

Scalability models: wrote 13 files


## 2. Scalability benchmark

Per model: measure wall-clock for load + trace extraction + structural
self-similarity. We assert self-similarity is exactly 1.0 as a sanity
check that the pipeline ran end-to-end.

The two shapes stress different parts of the pipeline:

- **Linear chain** — polynomial growth in element count, single trace
  variant. Stresses loading, set construction, normalization. We expect
  smooth scaling all the way to N=200.
- **Parallel AND** — element count grows linearly but trace variants
  grow factorially (`N!`). This is the intentional fidelity/cost
  trade-off of behavioral similarity; the wall appears around N=7–8
  (active-set cap engages, truncating from the theoretical N! variants).

In [5]:
LINEAR_SIZES = (5, 10, 20, 50, 100, 200)
AND_SIZES = (2, 3, 4, 5, 6, 7, 8)


def time_pipeline(path: Path, timeout_s: float = 120.0) -> dict:
    # Returns load / trace / sim wall-clock + variant count for one model.
    t0 = time.perf_counter()
    model = load_model(path)
    t_load = time.perf_counter() - t0

    t0 = time.perf_counter()
    res = extract_traces(model, timeout_seconds=timeout_s, max_loop_depth=3)
    t_trace = time.perf_counter() - t0

    t0 = time.perf_counter()
    sim = calculate_bpmn_similarity(model, model, method="dice")
    t_sim = time.perf_counter() - t0

    assert abs(sim["overall"] - 1.0) < 1e-9, (
        f"self-similarity drift on {path.name}: got {sim['overall']}"
    )
    return {
        "n": int(path.stem.rsplit("_", 1)[1]),
        "load_s": t_load,
        "trace_s": t_trace,
        "sim_s": t_sim,
        "total_s": t_load + t_trace + t_sim,
        "variants": len(res.variants),
        "is_sound": res.is_sound,
    }


def bench(prefix: str, sizes) -> pd.DataFrame:
    rows = []
    for n in sizes:
        path = SCALABILITY_DIR / f"{prefix}_{n}.bpmn"
        if not path.exists():
            print(f"  missing: {path.relative_to(REPO_ROOT)} (run section 1 first)")
            continue
        rows.append(time_pipeline(path))
    df = pd.DataFrame(rows).set_index("n")
    return df

In [6]:
linear_df = bench("linear", LINEAR_SIZES)
linear_df

,load_s,trace_s,sim_s,total_s,variants,is_sound
n,,,,,,
5,0.001280,0.003671,0.000117,0.005068,1,True
10,0.000889,0.001171,0.000109,0.002169,1,True
20,0.005786,0.003215,0.000192,0.009194,1,True
50,0.005746,0.012635,0.000473,0.018854,1,True
100,0.017724,0.033887,0.001120,0.052731,1,True
200,0.002608,0.112400,0.002946,0.117954,1,True


In [7]:
and_df = bench("and", AND_SIZES)
and_df

Trace extraction recovered partial results for net '<unnamed>': 27389 sound variant(s), 0 partial trace(s); exploration truncated by active-set cap


,load_s,trace_s,sim_s,total_s,variants,is_sound
n,,,,,,
2,0.000732,0.000685,0.000069,0.001486,2,True
3,0.000410,0.001063,0.000073,0.001546,6,True
4,0.000385,0.001981,0.000076,0.002441,24,True
5,0.000442,0.008452,0.000092,0.008986,120,True
6,0.000535,0.047447,0.000091,0.048074,720,True
7,0.000780,0.439127,0.000100,0.440008,5040,True
8,0.000939,2.442643,0.000126,2.443707,27389,False


**Headline numbers for the paper** — pulled from the two tables above.

The AND series at `N=8` typically hits the explorer's active-set cap; the
variants column shows the truncated count (often ~27k instead of the
theoretical 40,320 = 8!). That's expected and worth surfacing honestly in
the paper.

In [8]:
# A two-row summary suitable for pasting into the paper sentence.
summary = pd.DataFrame({
    "shape": ["linear chain", "parallel AND"],
    "max N": [linear_df.index.max(), and_df.index.max()],
    "wall-clock @ max N (s)": [
        f"{linear_df.loc[linear_df.index.max(), 'total_s']:.3f}",
        f"{and_df.loc[and_df.index.max(), 'total_s']:.3f}",
    ],
    "variants @ max N": [
        int(linear_df.loc[linear_df.index.max(), "variants"]),
        int(and_df.loc[and_df.index.max(), "variants"]),
    ],
})
summary

,shape,max N,wall-clock @ max N (s),variants @ max N
0,linear chain,200,0.118,1
1,parallel AND,8,2.444,27389


## 3. Edge-case pair scores

Every pair the maturity test suite exercises, with the actual
sub-scores measured under the current converter, normalizer, and
embedding model. The notebook does *not* assert thresholds here — it
just prints them, so you can see at a glance whether the pytest
assertions still have a healthy margin or whether something has
drifted.

In [9]:
# (category, left_relpath_under_examples/maturity, right_relpath, note)
EDGE_PAIRS = [
    ("sanity", "sanity/identical_baseline.bpmn", "sanity/identical_baseline.bpmn",
     "identical — expect ~1.0 everywhere"),
    ("sanity", "sanity/disjoint_left_credit.bpmn", "sanity/disjoint_right_student.bpmn",
     "disjoint — expect low overall"),
    ("sanity", "sanity/renamed_only_a.bpmn", "sanity/renamed_only_b.bpmn",
     "renamed only — raw modest, normalized should jump"),
    ("gateway", "gateway_substitutions/gateway_and.bpmn", "gateway_substitutions/gateway_xor.bpmn",
     "different gateway type, shared domain"),
    ("gateway", "gateway_substitutions/gateway_and.bpmn", "gateway_substitutions/gateway_or.bpmn",
     "different gateway type, shared domain"),
    ("gateway", "gateway_substitutions/gateway_xor.bpmn", "gateway_substitutions/gateway_or.bpmn",
     "different gateway type, shared domain"),
    ("structural", "structural_perturbations/linear_baseline.bpmn",
     "structural_perturbations/linear_reorder.bpmn",
     "small perturbation on linear sequence"),
    ("structural", "structural_perturbations/linear_baseline.bpmn",
     "structural_perturbations/linear_drift.bpmn",
     "larger perturbation on linear sequence"),
    ("structural", "structural_perturbations/and_two_branches.bpmn",
     "structural_perturbations/and_three_branches.bpmn",
     "branch added"),
    ("semantic", "semantic_naming/paraphrase_a.bpmn", "semantic_naming/paraphrase_b.bpmn",
     "paraphrase — relies on normalization"),
    ("semantic", "semantic_naming/synonym_a.bpmn", "semantic_naming/synonym_b.bpmn",
     "synonym — relies on normalization"),
    ("subprocess", "subprocess_folding/flat.bpmn", "subprocess_folding/with_subprocess.bpmn",
     "flat vs expanded subprocess — same labels"),
    ("round_trip", "format_round_trip/linear_sequence.bpmn",
     "format_round_trip/linear_sequence.json",
     "BPMN vs Signavio JSON"),
    ("round_trip", "format_round_trip/credit.bpmn", "format_round_trip/credit.json",
     "BPMN vs Signavio JSON"),
    ("degenerate", "degenerate/unsound_and_no_join.bpmn",
     "degenerate/sound_and_with_join.bpmn",
     "unsound vs sound — must not raise"),
    ("degenerate", "degenerate/empty.bpmn", "degenerate/empty.bpmn",
     "empty self — finite, no crash"),
    ("degenerate", "degenerate/single_task.bpmn", "degenerate/single_task.bpmn",
     "single-task self — expect 1.0"),
    ("degenerate", "degenerate/empty.bpmn", "degenerate/single_task.bpmn",
     "empty vs single-task — finite"),
]


def safe_trace_similarity(model_a, model_b) -> Optional[float]:
    try:
        res_a = extract_traces(model_a, timeout_seconds=10.0, max_loop_depth=3)
        res_b = extract_traces(model_b, timeout_seconds=10.0, max_loop_depth=3)
        return calculate_trace_similarity(res_a, res_b, method="jaccard")
    except Exception:  # surfaced via the cell output, no need to crash the notebook
        return None

In [10]:
rows = []
for category, left_name, right_name, note in EDGE_PAIRS:
    left_path = MATURITY_DIR / left_name
    right_path = MATURITY_DIR / right_name
    if not left_path.exists() or not right_path.exists():
        rows.append({
            "category": category, "left": left_name, "right": right_name,
            "overall": None, "elements": None, "trace": None,
            "note": f"MISSING — {note}",
        })
        continue
    left = load_model(left_path)
    right = load_model(right_path)
    struct = calculate_bpmn_similarity(left, right, method="dice")
    rows.append({
        "category": category,
        "left": left_name,
        "right": right_name,
        "overall": struct["overall"],
        "elements": struct["high_level_scores"].get("elements"),
        "trace": safe_trace_similarity(left, right),
        "note": note,
    })

edge_df = pd.DataFrame(rows)
# Display with rounded scores for readability.
edge_df_display = edge_df.copy()
for col in ("overall", "elements", "trace"):
    edge_df_display[col] = edge_df_display[col].apply(
        lambda v: f"{v:.3f}" if isinstance(v, float) else v
    )
edge_df_display

Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 34561 partial trace(s); 35047 loop-cap hit(s)


Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 30 partial trace(s); 1 distinct deadlock marking(s)


Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 4 partial trace(s); 1 distinct deadlock marking(s)


,category,left,right,overall,elements,trace,note
0,sanity,sanity/identical_baseline.bpmn,sanity/identical_baseline.bpmn,1.000,1.000,1.000,identical — expect ~1.0 everywhere
1,sanity,sanity/disjoint_left_credit.bpmn,sanity/disjoint_right_student.bpmn,0.087,0.291,0.000,disjoint — expect low overall
2,sanity,sanity/renamed_only_a.bpmn,sanity/renamed_only_b.bpmn,0.281,0.750,0.000,"renamed only — raw modest, normalized should jump"
3,gateway,gateway_substitutions/gateway_and.bpmn,gateway_substitutions/gateway_xor.bpmn,0.325,0.500,0.000,"different gateway type, shared domain"
4,gateway,gateway_substitutions/gateway_and.bpmn,gateway_substitutions/gateway_or.bpmn,0.413,0.708,0.000,"different gateway type, shared domain"
5,gateway,gateway_substitutions/gateway_xor.bpmn,gateway_substitutions/gateway_or.bpmn,0.325,0.500,0.000,"different gateway type, shared domain"
6,structural,structural_perturbations/linear_baseline.bpmn,structural_perturbations/linear_reorder.bpmn,0.268,0.714,0.000,small perturbation on linear sequence
7,structural,structural_perturbations/linear_baseline.bpmn,structural_perturbations/linear_drift.bpmn,0.174,0.464,0.000,larger perturbation on linear sequence
8,structural,structural_perturbations/and_two_branches.bpmn,structural_perturbations/and_three_branches.bpmn,0.597,0.822,0.000,branch added
9,semantic,semantic_naming/paraphrase_a.bpmn,semantic_naming/paraphrase_b.bpmn,0.281,0.750,0.000,paraphrase — relies on normalization


### Normalization on the renamed pair

The "renamed only" claim is the canonical maturity story: same shape,
different label strings. Raw similarity should be modest; after
`normalize_atomic_names` aligns the second model's vocabulary to the
first's, the score should jump close to 1.0.

In [11]:
from bpmn_normalization import normalize_atomic_names
from utils.string_similarity import cosine_sim_optimized

renamed_pairs = [
    ("renamed_only",
     MATURITY_DIR / "sanity" / "renamed_only_a.bpmn",
     MATURITY_DIR / "sanity" / "renamed_only_b.bpmn"),
    ("paraphrase",
     MATURITY_DIR / "semantic_naming" / "paraphrase_a.bpmn",
     MATURITY_DIR / "semantic_naming" / "paraphrase_b.bpmn"),
    ("synonym",
     MATURITY_DIR / "semantic_naming" / "synonym_a.bpmn",
     MATURITY_DIR / "semantic_naming" / "synonym_b.bpmn"),
    ("disjoint",
     MATURITY_DIR / "sanity" / "disjoint_left_credit.bpmn",
     MATURITY_DIR / "sanity" / "disjoint_right_student.bpmn"),
]

rows = []
for name, a_path, b_path in renamed_pairs:
    a = load_model(a_path)
    b = load_model(b_path)
    raw = calculate_bpmn_similarity(a, b, method="dice")["overall"]
    aligned, _ = normalize_atomic_names(a, b, cosine_sim_optimized, threshold=0.7)
    norm = calculate_bpmn_similarity(a, aligned, method="dice")["overall"]
    rows.append({"pair": name, "raw": f"{raw:.3f}", "normalized": f"{norm:.3f}"})

pd.DataFrame(rows)

,pair,raw,normalized
0,renamed_only,0.281,1.000
1,paraphrase,0.281,1.000
2,synonym,0.281,1.000
3,disjoint,0.087,0.087


The expected pattern: `renamed_only`, `paraphrase`, and `synonym` should
all sit around the same raw score (~0.28) and jump to ~1.0 after
normalization; `disjoint` should stay low both before and after. That's
the evidence behind the paper's maturity claim about semantic
normalization.